# ARK-020 V4 — hardened A1.2 operator launcher

This launcher preserves the current Colab runtime. Cell 0 mounts/validates Drive, refreshes the repository to the pinned A1.2 executable, and performs the fail-closed scan. Cell 1 runs the CPU contract suites in an isolated subprocess while keeping the T4 attached. Cell 2 starts/resumes the real CUDA campaign only after both gates pass.

In [ ]:
import json, subprocess, sys, os, shutil
from pathlib import Path

PINNED_RUNNER_COMMIT = 'e99aa0eb65404a6a19ed3926f0d3bae8da02194f'
BRANCH = 'codex/arkenstone-v4-durability-hardening'
REPO = '/content/An-Ra-the-new-AGI-ark020v4'
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
SCAN_GATE_PASS = False
SAFE_ACTION = None

print('=== ARK-020 V4 A1.2 CELL 0: MOUNT / PIN / FAIL-CLOSED SCAN ===')
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = Path('/content/drive/MyDrive').is_dir()
except Exception as exc:
    DRIVE_OK = False
    print('DRIVE MOUNT FAILED:', repr(exc))
if not DRIVE_OK:
    raise SystemExit('SAFE ACTION: STOP — DRIVE UNAVAILABLE')

if os.path.exists(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    shutil.rmtree(REPO)
if not os.path.exists(REPO):
    subprocess.run(['git','clone','--depth','100','--branch',BRANCH,REPO_URL,REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'remote','set-url','origin',REPO_URL], check=True)
    subprocess.run(['git','-C',REPO,'fetch','--depth','100','origin',BRANCH], check=True)

subprocess.run(['git','-C',REPO,'reset','--hard'], check=True)
subprocess.run(['git','-C',REPO,'clean','-fd'], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
print('PINNED A1.2 COMMIT OK:', head)

runner = os.path.join(REPO, 'experiments/ARK-020-V4/run_ark020_v4_hardened.py')
scan = subprocess.run([sys.executable, runner, '--mode','scan','--drive-ok','True'], cwd=REPO, capture_output=True, text=True)
print(scan.stdout)
if scan.stderr.strip():
    print('--- scan stderr ---')
    print(scan.stderr)
if scan.returncode != 0:
    raise SystemExit(f'SAFE ACTION: STOP — SCAN COMMAND FAILED ({scan.returncode})')
marker = '@@SCAN_JSON@@'
marker_line = next((ln for ln in scan.stdout.splitlines() if ln.startswith(marker)), None)
if marker_line is None:
    raise SystemExit('SAFE ACTION: STOP — SCAN DID NOT EMIT @@SCAN_JSON@@')
scan_info = json.loads(marker_line[len(marker):])
SAFE_ACTION = scan_info.get('SAFE_ACTION')
print('PARSED SAFE ACTION:', SAFE_ACTION)
if SAFE_ACTION not in {'START NEW CAMPAIGN', 'RESUME'}:
    raise SystemExit('Not safe to continue: ' + str(SAFE_ACTION))
SCAN_GATE_PASS = True
print('CELL 0 GATE: PASS')

In [ ]:
import subprocess, sys, torch, os

assert globals().get('SCAN_GATE_PASS') is True, 'Run Cell 0 successfully first.'
TEST_GATE_PASS = False
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before running.'
print('ATTACHED GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

compile_paths = [
    'experiments/ARK-020-V4/ark020_v4_core.py',
    'experiments/ARK-020-V4/run_ark020_v4.py',
    'experiments/ARK-020-V4/ark020_v4_durability.py',
    'experiments/ARK-020-V4/ark020_v4_a1_guardrails.py',
    'experiments/ARK-020-V4/ark020_v4_device_guard.py',
    'experiments/ARK-020-V4/run_ark020_v4_hardened.py',
    'tests/conftest.py',
    'tests/test_ark020_v4.py',
    'tests/test_ark020_v4_durability.py',
    'tests/test_ark020_v4_a1_guardrails.py',
]
for p in compile_paths:
    subprocess.run([sys.executable, '-m', 'py_compile', os.path.join(REPO, p)], check=True)
print('compile gate: PASS on', len(compile_paths), 'files')

test_env = dict(os.environ)
test_env['CUDA_VISIBLE_DEVICES'] = ''
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
test_env['PYTHONPATH'] = REPO + os.pathsep + test_env.get('PYTHONPATH', '')

print('\nRunning A1/A1.1/A1.2 durability contracts (CPU-isolated)...')
a1 = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_ark020_v4_durability.py', 'tests/test_ark020_v4_a1_guardrails.py', '-q'], cwd=REPO, env=test_env)
if a1.returncode != 0:
    raise SystemExit('Durability contract suite failed — DO NOT RUN campaign')

print('\nRunning inherited V4 suite (CPU-isolated)...')
v4 = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_ark020_v4.py', '-q'], cwd=REPO, env=test_env)
if v4.returncode != 0:
    raise SystemExit('Inherited V4 suite failed — DO NOT RUN campaign')

assert torch.cuda.is_available(), 'T4 disappeared after tests.'
print('GPU STILL ATTACHED:', torch.cuda.get_device_name(0))
TEST_GATE_PASS = True
print('CELL 1 GATE: PASS — durability + inherited V4 contracts passed; T4 preserved')

In [ ]:
# Full campaign. Exact-resumable. This uses the normal CUDA-visible environment.
import os, subprocess, sys, json, torch
from pathlib import Path

assert globals().get('SCAN_GATE_PASS') is True, 'Cell 0 gate not passed.'
assert globals().get('TEST_GATE_PASS') is True, 'Cell 1 gate not passed.'
assert torch.cuda.is_available(), 'T4 is not available.'
runner = os.path.join(REPO, 'experiments/ARK-020-V4/run_ark020_v4_hardened.py')
root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
env = dict(os.environ)
env.pop('CUDA_VISIBLE_DEVICES', None)
env['PYTHONUNBUFFERED'] = '1'
print('=== STARTING / RESUMING ARK-020 V4 A1.2 ON', torch.cuda.get_device_name(0), '===', flush=True)
proc = subprocess.run([sys.executable, runner, '--mode','all'], cwd=REPO, env=env)
print('CAMPAIGN RETURN CODE:', proc.returncode)

if proc.returncode != 0:
    failure = root / 'ARK-020_V4_FAILURE.json'
    if failure.exists():
        print('\n===== REAL V4 FAILURE RECEIPT =====')
        try:
            f = json.loads(failure.read_text())
            print('EXCEPTION:', f.get('exception'))
            print('MESSAGE:', f.get('message'))
            print(f.get('traceback', failure.read_text()))
        except Exception:
            print(failure.read_text())
    raise SystemExit('ARK-020 V4 child process failed — failure receipt printed above')

result = root / 'ARK-020_V4_RESULT.json'
session = root / 'SESSION_STATE.json'
if result.exists():
    r = json.loads(result.read_text())
    print('SCIENTIFIC CAMPAIGN STATUS:', r.get('status'))
    print('VERDICT:', r.get('decision', {}).get('verdict', r.get('verdict')))
elif session.exists():
    s = json.loads(session.read_text())
    print('SESSION STATUS:', s.get('status'))
    print('MESSAGE:', s.get('message'))
    if s.get('status') == 'PARTIAL_SESSION':
        print('Expected multi-session stop. On the next T4: rerun Cell 0 → Cell 1 → Cell 2.')

In [ ]:
from pathlib import Path
import json

root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
print('CAMPAIGN ROOT:', root)
for name in ['EXECUTABLE_IDENTITY_A1.json','PREEXECUTION_GATE.json','EXACT_RESUME_SMOKE_V4.json','EXACT_RESUME_SMOKE_V4_A1.json','SESSION_STATE.json','ARK-020_V4_RESULT.json','ARK-020_V4_FAILURE.json']:
    p = root / name
    if p.exists():
        print('\n===', name, '===')
        print(p.read_text()[:8000])